# Capstone: heavy experiments on Kaggle

Runs what is too heavy for the laptop, and downloads UK-DALE at data-centre speed:

1. **REFIT, deep forecasters** (GPU): the CNNs and BiLSTMs of Yaprakdal (2023), the GRU and CNN-GRU of
   Yaprakdal (2024), with the classical forecasters and the hourly ensemble, on 20 homes.
2. **UK-DALE**: download (3.6 GB), prepare with the same rules as REFIT, then the habit models, the
   suggestions (with London's real solar) and the forecasters.
3. **External test**: models trained on REFIT (Loughborough), tested on UK-DALE (London).
4. Everything small is packed into `results.zip` (Output tab) to bring back to the laptop.

**Settings (right-hand panel):** Accelerator = GPU (T4 or P100), Internet = On (needs a phone-verified
account), and the dataset `capstone-bundle` added under Input.

**To run with the laptop closed:** Save Version > Save & Run All (Commit). Each step writes a log to
`logs/`; a failing step does not stop the others.

In [ ]:
import glob, os, shutil, subprocess
# os.environ['NTFY_URL'] = 'https://ntfy.sh/<your-topic>'   # used only if the NTFY_URL secret is missing
found = glob.glob('/kaggle/input/**/habits', recursive=True)
assert found, 'No bundle: add the capstone-bundle dataset with Add Input (right-hand panel), then rerun'
bundle = os.path.dirname(found[0])
shutil.copytree(bundle, '/kaggle/working/capstone', dirs_exist_ok=True)
os.chdir('/kaggle/working/capstone')
os.makedirs('logs', exist_ok=True)
print('bundle from', bundle)
!pip install -q paho-mqtt
import torch, sklearn, pandas, sys
print('python', sys.version.split()[0], '| torch', torch.__version__, '| sklearn', sklearn.__version__,
      '| pandas', pandas.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (check the Accelerator setting)')

## 1. REFIT: deep forecasters (GPU)

In [ ]:
!python -m pytest -q tests/test_energy.py 2>&1 | tail -3
!python scripts/eval_forecast.py --deep 2>&1 | tee logs/refit_forecast_deep.log | tail -30

## 2. UK-DALE: download and prepare

In [ ]:
!mkdir -p /kaggle/temp && wget -q -c -O /kaggle/temp/ukdale.zip "https://dap.ceda.ac.uk/edc/d1/7d78f943-f9fe-413b-af52-1816f9d968b0/data/version_0/ukdale.zip?download=1"
!ls -l /kaggle/temp/ukdale.zip   # should be 3585155959 bytes
!CAPSTONE_DATASET=ukdale python scripts/prepare_ukdale.py /kaggle/temp/ukdale.zip 2>&1 | tee logs/ukdale_prepare.log | tail -60

## 3. UK-DALE: the same experiments as on REFIT

In [ ]:
!CAPSTONE_DATASET=ukdale python scripts/eval_refit.py 2>&1 | tee logs/ukdale_habit_models.log | tail -40
!CAPSTONE_DATASET=ukdale python scripts/eval_suggestions.py 2>&1 | tee logs/ukdale_suggestions.log | tail -40
!CAPSTONE_DATASET=ukdale python scripts/eval_forecast.py 2>&1 | tee logs/ukdale_forecast.log | tail -40
!CAPSTONE_DATASET=ukdale python scripts/eval_forecast.py --deep 2>&1 | tee logs/ukdale_forecast_deep.log | tail -30

## 4. External test: trained on REFIT, tested on UK-DALE

In [ ]:
!python scripts/eval_external.py 2>&1 | tee logs/external.log | tail -40

## 5. Pack the results (download `results.zip` from the Output tab)

In [ ]:
import zipfile, pathlib
with zipfile.ZipFile('/kaggle/working/results.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in ['data/results', 'data/processed/ukdale', 'data/raw/pvgis', 'logs']:
        for f in pathlib.Path(folder).rglob('*'):
            if f.is_file():
                zf.write(f, str(f))
print(round(pathlib.Path('/kaggle/working/results.zip').stat().st_size / 1e6, 1), 'MB')
shutil.rmtree('/kaggle/working/capstone/data/processed/refit', ignore_errors=True)  # keep the Output small